In [ ]:
# Load corpus
with open("/content/data.txt", "r", encoding="utf-8") as file:
    text = file.read()

# Load tokenizer
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file(
    "/content/ai_tokenizer.json"
)

print("Corpus characters:", len(text))
print("Vocabulary size:", tokenizer.get_vocab_size())

Corpus characters: 30220
Vocabulary size: 1589


In [ ]:
# Convert text into tokens and token IDs
encoding = tokenizer.encode(text)

token_ids = encoding.ids

print("Total tokens:", len(token_ids))
print("First 20 token IDs:", token_ids[:20])

Total tokens: 6920
First 20 token IDs: [561, 309, 9, 101, 10, 100, 622, 149, 459, 852, 94, 707, 97, 1062, 207, 192, 12, 811, 971, 222]


In [ ]:
import torch

# Convert token IDs into a PyTorch tensor
token_ids = torch.tensor(token_ids, dtype=torch.long)

print("Token IDs:", token_ids)
print("Shape:", token_ids.shape)
print("Data type:", token_ids.dtype)

Token IDs: tensor([561, 309,   9,  ..., 573, 719,  14])
Shape: torch.Size([6920])
Data type: torch.int64


In [ ]:
# Define context window
block_size = 8

# Split corpus into training and validation data
split_index = int(0.9 * len(token_ids))

train_data = token_ids[:split_index]
val_data = token_ids[split_index:]

print("Total tokens:", len(token_ids))
print("Training tokens:", len(train_data))
print("Validation tokens:", len(val_data))
print("Block size:", block_size)

Total tokens: 6920
Training tokens: 6228
Validation tokens: 692
Block size: 8


In [ ]:
def get_batch(data, batch_size=16, block_size=8):

    # Select random starting positions
    ix = torch.randint(
        len(data) - block_size - 1,
        (batch_size,)
    )

    # Input sequences
    x = torch.stack([
        data[i:i + block_size] for i in ix
    ])

    # Target sequences (shifted by one token)
    y = torch.stack([
        data[i + 1:i + block_size + 1] for i in ix
    ])

    return x, y

In [ ]:
batch_size = 16

x, y = get_batch(train_data, batch_size, block_size)

print("Input shape:", x.shape)
print("Target shape:", y.shape)

Input shape: torch.Size([16, 8])
Target shape: torch.Size([16, 8])


In [ ]:
import torch.nn as nn
import torch.nn.functional as F

# Model configuration
vocab_size = tokenizer.get_vocab_size()
embedding_dim = 32
hidden_size = 128

print("Vocabulary size:", vocab_size)
print("Embedding dimension:", embedding_dim)
print("Hidden layer size:", hidden_size)

Vocabulary size: 1589
Embedding dimension: 32
Hidden layer size: 128


In [ ]:
class SimpleNLM(nn.Module):

    def __init__(self, vocab_size, block_size, embedding_dim, hidden_size):
        super().__init__()

        # Token embedding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)

        # Neural network layers
        self.fc1 = nn.Linear(block_size * embedding_dim, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):

        # Convert token IDs into embeddings
        x = self.embedding(x)

        # Flatten embeddings
        x = x.reshape(x.size(0), -1)

        # Hidden layer
        x = self.relu(self.fc1(x))

        # Predict next token scores
        logits = self.fc2(x)

        return logits

In [ ]:
model = SimpleNLM(
    vocab_size=vocab_size,
    block_size=block_size,
    embedding_dim=embedding_dim,
    hidden_size=hidden_size
)

print(model)

SimpleNLM(
  (embedding): Embedding(1589, 32)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1589, bias=True)
)


In [ ]:
# Pass input batch through the model
logits = model(x)

print("Input shape:", x.shape)
print("Output shape:", logits.shape)

Input shape: torch.Size([16, 8])
Output shape: torch.Size([16, 1589])


In [ ]:
def get_batch(data, batch_size=16, block_size=64):

    ix = torch.randint(len(data) - block_size - 1, (batch_size,))

    x = torch.stack([data[i:i + block_size] for i in ix])

    # One next token after the 64-token context
    y = torch.stack([data[i + block_size] for i in ix])

    return x, y

In [ ]:
x, y = get_batch(train_data, batch_size, block_size)

logits = model(x)

print("Logits shape:", logits.shape)
print("Target shape:", y.shape)

Logits shape: torch.Size([16, 1589])
Target shape: torch.Size([16])


In [ ]:
loss = F.cross_entropy(logits, y)

print("Cross Entropy Loss:", loss.item())

Cross Entropy Loss: 7.410332202911377


In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [ ]:
# 1. Clear previous gradients
optimizer.zero_grad()

# 2. Forward pass
logits = model(x)

# 3. Calculate loss
loss = F.cross_entropy(logits, y)

# 4. Backpropagation
loss.backward()

# 5. Update model weights
optimizer.step()

print("Training Loss:", loss.item())

Training Loss: 7.410332202911377


In [ ]:
learning_rate = 0.001
max_iterations = 500
batch_size = 16

In [ ]:
for iteration in range(max_iterations):

    # Get a training batch
    x, y = get_batch(train_data, batch_size, block_size)

    # Clear old gradients
    optimizer.zero_grad()

    # Forward pass
    logits = model(x)

    # Calculate training loss
    loss = F.cross_entropy(logits, y)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Display loss
    if iteration % 50 == 0:
        print(f"Iteration: {iteration}, Loss: {loss.item():.4f}")

Iteration: 0, Loss: 7.3847
Iteration: 50, Loss: 6.7556
Iteration: 100, Loss: 5.5572
Iteration: 150, Loss: 5.7199
Iteration: 200, Loss: 6.5026
Iteration: 250, Loss: 5.6667
Iteration: 300, Loss: 4.3013
Iteration: 350, Loss: 5.6465
Iteration: 400, Loss: 5.3378
Iteration: 450, Loss: 6.0030


In [ ]:
def evaluate(model, data, batches=20):

    model.eval()
    total_loss = 0

    with torch.no_grad():

        for _ in range(batches):

            x, y = get_batch(data, batch_size, block_size)

            logits = model(x)

            loss = F.cross_entropy(logits, y)

            total_loss += loss.item()

    model.train()

    return total_loss / batches


print("Training Loss:", evaluate(model, train_data))
print("Validation Loss:", evaluate(model, val_data))

Training Loss: 5.461912298202515
Validation Loss: 6.6461179733276365


In [ ]:
model.eval()

total_loss = 0

with torch.no_grad():
    for _ in range(20):

        val_x, val_y = get_batch(
            val_data,
            batch_size,
            block_size
        )

        val_logits = model(val_x)

        loss = F.cross_entropy(val_logits, val_y)

        total_loss += loss.item()

avg_val_loss = total_loss / 20

print("Average Validation Loss:", avg_val_loss)

model.train()

Average Validation Loss: 6.85706672668457


SimpleNLM(
  (embedding): Embedding(1589, 32)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=128, out_features=1589, bias=True)
)

In [ ]:
def predict_next_token(sentence):

    # Convert sentence into token IDs
    ids = tokenizer.encode(sentence).ids

    if len(ids) < block_size:
        print(f"Please enter at least {block_size} tokens.")
        print("Your input contains:", len(ids))
        return

    # Take the last 64 tokens
    input_ids = torch.tensor(
        ids[-block_size:],
        dtype=torch.long
    ).unsqueeze(0)

    # Predict next token
    model.eval()

    with torch.no_grad():
        logits = model(input_ids)

        predicted_id = torch.argmax(logits, dim=1).item()

    model.train()

    # Convert predicted ID back into text
    predicted_token = tokenizer.decode([predicted_id])

    print("Input sentence:", sentence)
    print("Predicted next token:", predicted_token)

In [ ]:
sentence = """Generative AI code generation tools and automation tool which """
predict_next_token(sentence)

Input sentence: Generative AI code generation tools and automation tool which 
Predicted next token: the
